# DMSTL: previsão em painel

Esta versão segue o fluxo de decomposição local por SKU, consolidação em painéis por tendência e sazonalidade e uso de um único modelo residual para todos os SKUs.

O objetivo é comparar o custo da estratégia local (um modelo residual por SKU) com a estratégia em painel (um único residual global + agregação por série), mantendo o mesmo esquema de saída.


In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / 'pyproject.toml').exists():
    if repo_root.parent == repo_root:
        raise RuntimeError('Nao foi possivel localizar o diretorio do projeto.')
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal
from sklearn.linear_model import LinearRegression
from statsforecast.models import AutoETS, SeasonalNaive
from mlforecast import MLForecast

from tinyshift.modelling import DMSTLWrapper
from dmstlv2 import DMSTLWrapperV2 

/home/heylucasleao/forecasting/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from functools import partial

In [3]:
# Copyright (c) 2024-2026 Lucas Leão
# tinyshift - A small toolbox for mlops
# Licensed under the MIT License

import copy
from functools import partial
from typing import Any, Callable, Dict, List, Literal, Optional, Tuple, Union
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin

from tinyshift.series import detect_seasonal_periods, select_pami_lag
from tinyshift.utils.imports import requires_extra


class DMSTLWrapperV(BaseEstimator, RegressorMixin):
    """
    Decomposed Multiple Seasonal-Trend (DMSTL) Global/Panel wrapper.

    Decomposes multi-seasonal time series locally per unique_id into trend,
    seasonal, and residual components using MSTL. Fits global statistical base
    models on trend and seasonal components via StatsForecast in parallel, while
    modeling complex non-linear residual dynamics globally using MLForecast.
    """

    @requires_extra("series")
    def __init__(
        self,
        residual_model_callable: Optional[
            Union[
                Callable[[List[int], Union[str, int]], Any],
                Dict[Union[str, int], Callable[[List[int], Union[str, int]], Any]],
            ]
        ] = None,
        freq: Optional[Union[str, int]] = None,
        season_length: Optional[
            Union[
                int,
                List[int],
                Dict[Union[str, int], Union[int, List[int], Literal["auto"]]],
                Literal["auto"],
            ]
        ] = "auto",
        seasonal_detection_params: Optional[Dict[str, Any]] = None,
        trend_model_callable: Optional[
            Union[
                Callable[[], Any],
                Dict[Union[str, int], Callable[[], Any]],
            ]
        ] = None,
        seasonal_model_callable: Optional[
            Union[
                Callable[[int], Any],
                Dict[Union[str, int], Callable[[int], Any]],
            ]
        ] = None,
        nlags: Optional[
            Union[
                int,
                List[int],
                Dict[Union[str, int], Union[int, List[int], Literal["auto"]]],
                Literal["auto"],
            ]
        ] = "auto",
        pami_params: Optional[Dict[str, Any]] = None,
        log_transform: bool = False,
        n_jobs: int = -1,
    ) -> None:
        self.residual_model_callable = residual_model_callable
        self.freq = freq
        self.season_length = season_length
        self.seasonal_detection_params = seasonal_detection_params or {}
        self.trend_model_callable = trend_model_callable
        self.seasonal_model_callable = seasonal_model_callable
        self.nlags = nlags
        self.pami_params = pami_params or {}
        self.log_transform = log_transform
        self.n_jobs = n_jobs

    def _get_sku_config(self, config, uid: Union[str, int]):
        if isinstance(config, dict):
            return config.get(uid)
        return config

    def _get_model_cols(self, df: pd.DataFrame) -> List[str]:
        return [c for c in df.columns if c not in [self.id_col_, self.time_col_]]

    def _process_components(
        self, components_df: pd.DataFrame
    ) -> Tuple[np.ndarray, Dict[int, np.ndarray], np.ndarray]:
        trend_part = components_df["trend"].bfill().ffill().values
        residual_part = components_df["resid"].fillna(0.0).values

        seasonal_dict = {}
        for col in components_df.columns:
            if col.startswith("seasonal_"):
                period = int(col.split("_")[1])
                seasonal_dict[period] = components_df[col].fillna(0.0).values

        return trend_part, seasonal_dict, residual_part

    def _resolve_seasonal_lengths(
        self, uid: Union[str, int], series: np.ndarray
    ) -> List[int]:
        season_length = self._get_sku_config(self.season_length, uid)

        if season_length is None:
            raise ValueError(
                f"No season_length configured for unique_id {uid!r}."
            )

        if season_length == "auto":
            detected_periods = detect_seasonal_periods(
                series, **self.seasonal_detection_params
            )
            if not detected_periods:
                raise ValueError(
                    f"Automatic seasonal detection failed for unique_id {uid!r}."
                )
            season_lengths = detected_periods
        else:
            season_lengths = (
                [season_length] if isinstance(season_length, int) else season_length
            )

        if (
            not isinstance(season_lengths, list)
            or not season_lengths
            or len(set(season_lengths)) != len(season_lengths)
            or any(
                not isinstance(p, int) or isinstance(p, bool) or p <= 1
                for p in season_lengths
            )
        ):
            raise ValueError(
                f"season_length for unique_id {uid!r} must contain positive "
                "integer periods > 1 without duplicates."
            )

        return season_lengths

    def _resolve_residual_lags(
        self, uid: Union[str, int], residual_part: np.ndarray
    ) -> List[int]:
        lags_config = self._get_sku_config(self.nlags, uid)

        if lags_config == "auto":
            selected_lag, _, _ = select_pami_lag(residual_part, **self.pami_params)
            if isinstance(selected_lag, int):
                lags_list = [selected_lag] if selected_lag > 0 else [1]
            else:
                lags_list = selected_lag
            return lags_list if lags_list else [1]

        if isinstance(lags_config, int):
            return list(range(1, lags_config + 1))
        elif isinstance(lags_config, list):
            return lags_config

        raise ValueError(f"Invalid lags configuration for unique_id {uid!r}: {lags_config}")

    def _apply_horizontal_stabilization(
        self, y_hat: np.ndarray, method: Literal["hpi", "hfi"], w_s: float
    ) -> np.ndarray:
        from tinyshift.series import hfi, hpi

        if method == "hpi":
            return hpi(y_hat, w_s=w_s)
        elif method == "hfi":
            return hfi(y_hat, w_s=w_s)
        else:
            raise ValueError(f"Invalid method '{method}'. Choose 'hpi' or 'hfi'.")

    @requires_extra("series")
    def fit(
        self,
        df: pd.DataFrame,
        id_col: str = "unique_id",
        time_col: str = "ds",
        target_col: str = "y",
        prediction_intervals: Optional[Any] = None,
        static_features: Optional[List[str]] = None,
    ) -> "DMSTLWrapper":
        from statsforecast import StatsForecast
        from statsforecast.models import AutoETS, SeasonalNaive
        from statsmodels.tsa.seasonal import MSTL

        from tinyshift.series import extract_mstl_components

        if self.freq is None:
            raise ValueError("Parameter 'freq' must be declared upon initialization.")

        self.freq_ = self.freq
        self.id_col_ = id_col
        self.time_col_ = time_col
        self.target_col_ = target_col
        self.exog_cols_ = [
            c for c in df.columns if c not in [id_col, time_col, target_col]
        ]

        trend_frames = []
        residual_frames = []
        seasonal_frames: Dict[int, List[pd.DataFrame]] = {}
        
        sku_lags_dict: Dict[Union[str, int], List[int]] = {}
        all_seasonal_periods = set()

        # 1. ETAPA LOCAL: Decomposição MSTL e Extração de Componentes
        for uid, group in df.groupby(id_col):
            group_sorted = group.sort_values(time_col).copy()
            y_series = group_sorted[target_col].values

            if self.log_transform:
                y_series = np.log1p(y_series)

            season_lengths = self._resolve_seasonal_lengths(uid, y_series)
            all_seasonal_periods.update(season_lengths)

            mstl = MSTL(y_series, periods=season_lengths)
            res = mstl.fit()
            components_df = extract_mstl_components(res, season_lengths)
            
            trend_part, seasonal_dict, residual_part = self._process_components(components_df)

            dates = group_sorted[time_col].values

            # Buffer de Tendência
            trend_frames.append(
                pd.DataFrame({id_col: uid, time_col: dates, target_col: trend_part})
            )

            # Buffer de Resíduos
            df_res = group_sorted[[id_col, time_col] + self.exog_cols_].copy()
            df_res[target_col] = residual_part
            residual_frames.append(df_res)

            # Registro dos Lags por SKU
            sku_lags_dict[uid] = self._resolve_residual_lags(uid, residual_part)

            # Buffers Sazonais temporários por SKU
            for p in season_lengths:
                if p not in seasonal_frames:
                    seasonal_frames[p] = []
                seasonal_frames[p].append(
                    pd.DataFrame({id_col: uid, time_col: dates, target_col: seasonal_dict[p]})
                )

        # Preenchimento com 0.0 para SKUs que não possuem determinado período p
        all_uids = df[id_col].unique()
        for p in all_seasonal_periods:
            uids_with_p = {frame[id_col].iloc[0] for frame in seasonal_frames[p]}
            missing_uids = set(all_uids) - uids_with_p

            for m_uid in missing_uids:
                group_sorted = df[df[id_col] == m_uid].sort_values(time_col)
                zero_series = np.zeros(len(group_sorted))
                seasonal_frames[p].append(
                    pd.DataFrame({
                        id_col: m_uid,
                        time_col: group_sorted[time_col].values,
                        target_col: zero_series,
                    })
                )

        # 2. ETAPA GLOBAL: StatsForecast (Tendência e Sazonalidades)
        df_trend_panel = pd.concat(trend_frames, ignore_index=True)
        
        default_trend_model = AutoETS(model="ZZN")
        self.sf_trend_ = StatsForecast(
            models=[default_trend_model], freq=self.freq_, n_jobs=self.n_jobs
        ).fit(df_trend_panel)

        self.sf_seasonal_dict_ = {}
        for p in all_seasonal_periods:
            df_seas_panel = pd.concat(seasonal_frames[p], ignore_index=True)
            seas_model = SeasonalNaive(season_length=p, alias=f"SeasonalNaive-{p}")
            self.sf_seasonal_dict_[p] = StatsForecast(
                models=[seas_model], freq=self.freq_, n_jobs=self.n_jobs
            ).fit(df_seas_panel)

        # 3. ETAPA GLOBAL: MLForecast nos Resíduos com Máscara
        df_residual_panel = pd.concat(residual_frames, ignore_index=True)
        
        max_lag = max(max(lags) for lags in sku_lags_dict.values())
        global_lags = list(range(1, max_lag + 1))

        if callable(self.residual_model_callable):
            try:
                self.mf_resid_ = self.residual_model_callable(nlags=global_lags, freq=self.freq_)
            except TypeError:
                self.mf_resid_ = self.residual_model_callable(global_lags, self.freq_)
        else:
            raise ValueError("residual_model_callable must be provided as a callable factory.")

        # Ajuste do MLForecast no Painel Completo
        self.mf_resid_.fit(
            df_residual_panel,
            id_col=id_col,
            time_col=time_col,
            target_col=target_col,
            prediction_intervals=prediction_intervals,
            static_features=static_features,
        )

        self.sku_lags_dict_ = sku_lags_dict
        self.max_lag_ = max_lag
        return self

    @requires_extra("series")
    def predict(
        self,
        h: int,
        X_df: Optional[pd.DataFrame] = None,
        level: Optional[List[Union[int, float]]] = None,
        stabilization_method: Optional[Literal["hpi", "hfi"]] = None,
        w_s: float = 0.0,
    ) -> pd.DataFrame:
        if not hasattr(self, "mf_resid_"):
            raise RuntimeError("The model must be fitted with .fit() before calling predict.")

        if self.exog_cols_ and X_df is None:
            raise ValueError(
                f"Model was fitted with exogenous features {self.exog_cols_}. Provide 'X_df'."
            )

        # 1. Posições Globais das Previsões (Métricas Estatísticas)
        df_trend_preds = self.sf_trend_.predict(h=h)
        trend_cols = self._get_model_cols(df_trend_preds)
        
        df_trend_preds["_trend_sum"] = df_trend_preds[trend_cols].sum(axis=1)
        df_trend_preds = df_trend_preds.sort_values([self.id_col_, self.time_col_])

        # Previsão Sazonal Unificada
        seasonal_sum_df = df_trend_preds[[self.id_col_, self.time_col_]].copy()
        seasonal_sum_df["_seas_sum"] = 0.0

        for p, sf_seas in self.sf_seasonal_dict_.items():
            df_seas_pred = sf_seas.predict(h=h).sort_values([self.id_col_, self.time_col_])
            seas_cols = self._get_model_cols(df_seas_pred)
            seasonal_sum_df["_seas_sum"] += df_seas_pred[seas_cols].sum(axis=1).values

        # Previsão de Resíduos (MLForecast)
        df_resid_preds = self.mf_resid_.predict(h=h, X_df=X_df, level=level)
        df_resid_preds = df_resid_preds.sort_values([self.id_col_, self.time_col_]).copy()

        model_cols = self._get_model_cols(df_resid_preds)

        trend_vals = df_trend_preds["_trend_sum"].values
        seas_vals = seasonal_sum_df["_seas_sum"].values

        # 2. Recombinação Vetorizada sem .merge()
        for col in model_cols:
            res = df_resid_preds[col].values + trend_vals + seas_vals

            if self.log_transform:
                res = np.expm1(res)

            if stabilization_method is not None and w_s > 0.0:
                # Estabilização agrupada por SKU na matriz vetorial
                temp_df = pd.DataFrame({self.id_col_: df_resid_preds[self.id_col_], "val": res})
                res = (
                    temp_df.groupby(self.id_col_)["val"]
                    .transform(
                        lambda g: self._apply_horizontal_stabilization(
                            g.values, method=stabilization_method, w_s=w_s
                        )
                    )
                    .values
                )

            df_resid_preds[col] = res

        return df_resid_preds

In [4]:
def make_panel(n_series=80, history=84, seed=42):
    rng = np.random.default_rng(seed)
    steps = np.arange(history)
    dates = pd.date_range("2020-01-01", periods=history, freq="D")
    frames = []

    for index in range(n_series):
        values = (
            40
            + index * 0.2
            + 0.08 * steps
            + 6 * np.sin(2 * np.pi * steps / 7 + index / 8)
            + rng.normal(scale=0.8, size=history)
        )
        frames.append(
            pd.DataFrame(
                {"unique_id": f"series-{index}", "ds": dates, "y": values}
            )
        )

    return pd.concat(frames, ignore_index=True)


def residual_model_callable(nlags, freq):
    return MLForecast(
        models=[LinearRegression()],
        lags=nlags,
        freq=freq,
    )


def seasonal_model_callable(period):
    return SeasonalNaive(season_length=period)


def trend_model_callable():
    return AutoETS(model="ZZN")


def build_model(model_class):
    return model_class(
        residual_model_callable=residual_model_callable,
        freq="D",
        season_length=[7],
        trend_model_callable=trend_model_callable,
        seasonal_model_callable=seasonal_model_callable,
        nlags=[1, 2, 7],
    )


panel = make_panel()
horizon = 14
traditional = build_model(DMSTLWrapper).fit(panel)
panel_model = build_model(DMSTLWrapperV2).fit(panel)

traditional_predictions = traditional.predict(h=horizon)
panel_predictions = panel_model.predict(h=horizon)

assert len(panel_predictions) == panel["unique_id"].nunique() * horizon
assert set(panel_predictions["unique_id"]) == set(panel["unique_id"])
assert set(traditional_predictions.columns) == set(panel_predictions.columns)
print("O modelo de painel produziu previsões para todos os SKUs com o mesmo esquema de saída.")


ValueError: Detected temporal gaps/missing dates in 79 series (e.g., ['series-1', 'series-10', 'series-11']). MSTL requires a continuous time grid at frequency 'D'. Preprocess your data using gap-filling methods (e.g., `utilsforecast.preprocessing.fill_gaps`) before calling `fit()`.

In [6]:
panel

,unique_id,ds,y
0,series-0,2020-01-01,40.243774
1,series-0,2020-01-02,43.939002
2,series-0,2020-01-03,46.609928
3,series-0,2020-01-04,43.595754
4,series-0,2020-01-05,36.155869
...,...,...,...
6715,series-79,2020-03-20,56.517299
6716,series-79,2020-03-21,61.858480
6717,series-79,2020-03-22,67.175992
6718,series-79,2020-03-23,66.678209


In [ ]:
traditional_predictions = traditional.predict(h=horizon)
panel_predictions = panel_model.predict(h=horizon)

assert len(panel_predictions) == panel["unique_id"].nunique() * horizon
assert set(panel_predictions["unique_id"]) == set(panel["unique_id"])
assert set(traditional_predictions.columns) == set(panel_predictions.columns)
print("Validação final do painel: previsões consistentes com o esquema do modelo local.")

Validação final do painel: previsões consistentes com o esquema do modelo local.


In [ ]:
from timeit import repeat


def benchmark(predict_callable, repeats=5, calls_per_repeat=3):
    timings = repeat(predict_callable, repeat=repeats, number=calls_per_repeat)
    return np.asarray(timings) / calls_per_repeat


traditional_times = benchmark(lambda: traditional.predict(h=horizon))
panel_times = benchmark(lambda: panel_model.predict(h=horizon))

traditional_median = np.median(traditional_times)
panel_median = np.median(panel_times)
gain_pct = 100 * (traditional_median - panel_median) / traditional_median

benchmark_result = pd.DataFrame(
    {
        "implementation": ["DMSTLWrapper (local)", "PanelDMSTLWrapper (fluxo.txt)"],
        "median_seconds": [traditional_median, panel_median],
        "mean_seconds": [traditional_times.mean(), panel_times.mean()],
        "gain_vs_traditional_pct": [0.0, gain_pct],
    }
)
benchmark_result

,implementation,median_seconds,mean_seconds,gain_vs_traditional_pct
0,DMSTLWrapper (local),2.112279,2.108928,0.000000
1,PanelDMSTLWrapper (fluxo.txt),0.326433,0.325766,84.545952


## Leitura do resultado

Um ganho positivo indica que o batch de painel reduziu o tempo de previsao. O ganho vem de reduzir de `n_series` modelos residuais para um unico `MLForecast`, e de consolidar as chamadas de tendencia e sazonalidade em poucos objetos `StatsForecast` com `n_jobs=-1`.

Essa arquitetura e apropriada quando todos os SKUs compartilham `freq`, `nlags`, periodos sazonais e classes de modelos. Se os SKUs exigirem configuracoes distintas, agrupe-os por configuracao e crie um painel por grupo. A decomposicao MSTL continua univariada no `statsmodels`, logo o loop de decomposicao permanece no treino.